# Scarcity Before the Spike

**Point-in-time machine learning for next-day ERCOT price extremes**

This notebook is the research walkthrough; tested implementation lives in `src/ercot_spikes/`. The decision is made at **09:00 CT on D−1**, and the target is whether the next-day HB_NORTH hourly real-time price exceeds **$100/MWh**.

## 1. Start with the clock, not the algorithm

A feature is admissible only if it was public before the decision. Fixed 48-hour-lead GFS forecast vintages are admissible; realized target-day weather is not. Realized price and native-load histories are shifted by at least 48 hours. Day-ahead price is excluded from the classifier and reserved for a later economic diagnostic.

![Research design](../reports/figures/research_design.svg)

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path('..') if Path.cwd().name == 'notebooks' else Path('.')
OUTPUT = ROOT / 'outputs' / 'benchmark'
manifest = json.loads((OUTPUT / 'run_manifest.json').read_text())
manifest

{'research_question': 'Can public information available before the day-ahead auction identify next-day HB_NORTH real-time price spikes?',
 'target': 'hourly RT price > $100/MWh',
 'model_selection': {'period': '2024',
  'metric': 'precision-recall AUC',
  'eligible_models': ['Gradient boosting',
   'Random forest',
   'Neural network',
   'Seasonal prior',
   'Logistic regression',
   'RBF SVM'],
  'selection_ineligible_diagnostics': ['Gradient boosting + lagged load']},
 'selected_on_2024_validation': 'Gradient boosting',
 'selected_features': ['temperature_2m_austin',
  'temperature_2m_dallas',
  'temperature_2m_houston',
  'temperature_2m_midland',
  'temperature_2m_sanantonio',
  'temp_mean',
  'temp_max',
  'temp_min',
  'cooling_degree',
  'heating_degree',
  'hour_sin',
  'hour_cos',
  'dow_sin',
  'dow_cos',
  'month_sin',
  'month_cos',
  'is_weekend',
  'rt_price_lag_48h',
  'rt_price_lag_72h',
  'rt_price_lag_168h',
  'rt_mean_7d',
  'rt_vol_7d',
  'rt_max_7d',
  'recent_spi

## 2. Treat the target as a rare event

Only 235 of 8,759 test hours (2.68%) cross the threshold. Ordinary accuracy would reward a useless always-negative classifier. The primary score is therefore precision–recall AUC, complemented by ROC-AUC, Brier score, log loss, and precision/recall among the 5% highest-risk hours.

In [2]:
validation = pd.read_csv(OUTPUT / 'validation_metrics.csv').set_index('model')
test = pd.read_csv(OUTPUT / 'test_metrics.csv').set_index('model')
comparison = validation[['pr_auc']].join(test[['pr_auc']], lsuffix='_validation', rsuffix='_test')
comparison.sort_values('pr_auc_validation', ascending=False).round(3)

,pr_auc_validation,pr_auc_test
model,,
Gradient boosting,0.084,0.095
Gradient boosting + lagged load,0.079,0.088
Random forest,0.076,0.086
Neural network,0.067,0.077
Seasonal prior,0.056,0.066
Logistic regression,0.053,0.061
RBF SVM,0.045,0.069


## 3. Compare model classes without breaking the lockbox

The candidates cover a smoothed seasonal prior, regularized logistic regression, RBF SVM, random forest, histogram gradient boosting, and a two-layer MLP. A named retrospective ablation also adds 48-hour-lagged native load; because annual load archives can contain later settlement revisions, it is selection-ineligible, and it also lowers validation PR-AUC. Gradient boosting wins 2024 validation PR-AUC and is frozen as the primary model; it also remains first in the locked 2025 ranking.

![Model comparison](../outputs/benchmark/figures/model_comparison.png)

## 4. Interpret the learned risk score

Permutation importance is computed on the untouched 2025 year. Time-of-day, city-level temperature forecasts, seven-day price volatility, and lagged prices dominate. The score is therefore best read as a nonlinear interaction between forecast physical stress and market memory.

![Feature importance](../outputs/benchmark/figures/feature_importance.png)

In [3]:
risk = pd.read_csv(OUTPUT / 'risk_deciles.csv').set_index('risk_decile')
risk[['predicted_risk', 'realized_spike_rate', 'mean_rt_minus_da']].round(4)

,predicted_risk,realized_spike_rate,mean_rt_minus_da
risk_decile,,,
1,0.0003,0.0000,-0.3890
2,0.0016,0.0000,-0.6257
3,0.0028,0.0034,-0.5326
4,0.0077,0.0046,-0.3477
5,0.0089,0.0160,0.2616
6,0.0093,0.0114,-0.7850
7,0.0142,0.0263,-0.5793
8,0.0211,0.0320,3.7149
9,0.0396,0.0685,-3.3704


## 5. Separate tail-risk prediction from alpha

The highest-risk decile realizes a 10.62% spike rate, versus 2.68% overall. Yet its mean RT−DA spread is negative, and the daily block-bootstrap interval for virtual-load spread lift lies below zero. The day-ahead market can price a risk more aggressively than the average real-time outcome even when the operational event remains forecastable.

![Risk lift](../outputs/benchmark/figures/risk_lift.png)

**Conclusion.** The model is useful as a scarcity screen, but the prespecified virtual-load direction is wrong in this test. Reversing the position after observing that result would be post hoc.

A separate extension therefore predicts `RT−DA` directly. Its virtual-supply direction, one-trade-per-day cap, −$3/MWh forecast threshold, $2/MWh hurdle, model, and inference gate were committed before 2026 outcomes were acquired. The 2026 point estimate is positive, but its block-bootstrap interval crosses zero and five days dominate the result. The disciplined label is *alpha candidate*, not established alpha.

![Prospective alpha audit](../reports/figures/alpha_audit.png)

In [ ]:
alpha_output = ROOT / 'outputs' / 'alpha'
alpha_metrics = pd.read_csv(alpha_output / 'alpha_metrics.csv').set_index('period')
alpha_baselines = pd.read_csv(alpha_output / 'alpha_baselines.csv').set_index('period')
display(alpha_metrics[['trades', 'mean_net_pnl_per_mwh', 'mean_net_pnl_ci_95_low',
                       'mean_net_pnl_ci_95_high', 'annualized_daily_sharpe',
                       'top_five_profit_days_share']].round(3))
display(alpha_baselines[['observed_mean_net_pnl', 'random_hour_same_days_mean',
                         'random_hour_same_days_one_sided_p',
                         'random_date_hour_equal_turnover_one_sided_p']].round(3))

In [4]:
# Full recomputation (downloads excluded from Git):
# !python scripts/download_data.py
# !ercot-spike-benchmark --config configs/experiment.toml --output outputs/benchmark
# !ercot-alpha-research --config configs/alpha_lockbox.toml --output outputs/alpha